# 02 - Import a dataset from Hugging Face

**adaption-devkit** is a *community, unofficial* toolkit. **Not affiliated with or endorsed by Adaption Labs.** Apache-2.0. Author: Aivaras Navardauskas (MANIFESTA), GitHub A1VARA5.

Notebook 01 used a local file. This one ingests directly from a **Hugging Face dataset**, maps columns, estimates, and runs.

Key difference from a local upload: **Hugging Face ingestion is asynchronous**. The server downloads and processes the files in the background, so you must **wait until ingestion finishes** before you adapt.

All outputs are **illustrative**; this notebook was authored, not executed.

## Setup

Same environment-variable auth as notebook 01. Never hardcode the host or key.

```bash
export ADAPTION_BASE_URL="https://api.prod.adaptionlabs.ai"
export ADAPTION_API_KEY="pt_live_..."
```

In [ ]:
%pip install -q adaption

In [ ]:
import os, time
from adaption import Adaption

BASE_URL = os.environ["ADAPTION_BASE_URL"]
assert os.environ.get("ADAPTION_API_KEY"), "Set ADAPTION_API_KEY in your environment first."
client = Adaption(base_url=BASE_URL)  # api_key from ADAPTION_API_KEY
print("Client ready.")

## Step 1 - Kick off the Hugging Face import

Point `create_from_huggingface` at a dataset URL and list the file(s) you want. Replace the placeholders below with a real public dataset that has a high quality **answers** column (so we can keep the completion only pattern from notebook 01).

Pick a dataset whose anchors are naturally unique. Remember: **deduplication is always on and keys on the prompt**. If you map a `prompt` column that repeats a templated instruction across rows, the set collapses. Completion only avoids that entirely.

In [ ]:
# Replace with a real dataset you have the rights to use.
HF_URL = "https://huggingface.co/datasets/your-org/your-marketing-corpus"
HF_FILES = ["train.csv"]  # the file(s) inside that repo to ingest

resp = client.datasets.create_from_huggingface(url=HF_URL, files=HF_FILES)
dataset_id = resp.dataset_id
print("dataset_id:", dataset_id)  # illustrative, e.g. 'ds_7b21...'

## Step 2 - Wait for ingestion (it is async)

A Hugging Face dataset is ready to adapt once `get_status(dataset_id).row_count` is no longer `None`. We poll until then. (If your SDK build exposes `wait_for_completion` for ingestion you can use that instead; the manual poll below works everywhere.)

In [ ]:
while True:
    status = client.datasets.get_status(dataset_id)
    if status.row_count is not None:
        break
    print("ingesting... row_count not ready yet")
    time.sleep(5)

print("ingestion done. row_count:", status.row_count)  # illustrative, e.g. 4200

## Step 3 - Inspect columns and build the mapping

You need to know the real column names in the imported dataset to map them. If you already know them from the HF dataset card, set them directly. Otherwise download a small slice to peek.

Below we assume the dataset has an `answer` column plus `channel` and `persona` context columns. **Adjust the names to your dataset.**

In [ ]:
# Optional: peek at the ingested rows by downloading and reading a few lines.
# Read with utf-8-sig so a BOM never corrupts the first column name.
import pandas as pd

try:
    peek_url = client.datasets.download(dataset_id)  # presigned URL
    peek = pd.read_csv(peek_url, encoding="utf-8-sig", nrows=5)
    print("columns:", list(peek.columns))
    display(peek)
except Exception as e:
    # download returns 422 if no run has started yet; that is fine here, we just
    # fall back to the column names from the dataset card.
    print("peek skipped:", e)

In [ ]:
# Map to YOUR dataset's real column names.
column_mapping = {
    "completion": "answer",            # the high quality answer column -> completion only anchor
    "context": ["channel", "persona"],  # context is always a list
    # If your dataset has a genuine, per-row-unique instruction column, you may instead use:
    #   "prompt": "instruction", "completion": "answer"
}
column_mapping

## Step 4 - Estimate, then pilot-run

Same discipline as notebook 01: **estimate first**, then a capped pilot.

In [ ]:
BLUEPRINT = (
    "Write in a warm, concrete, benefit-led marketing voice. "
    "Lead with the customer outcome, avoid hype and cliche, keep sentences crisp."
)
RECIPES = {"deduplication": True, "prompt_rephrase": True}
BRAND = {"length": "concise", "blueprint": BLUEPRINT}

estimate = client.datasets.run(
    dataset_id,
    column_mapping=column_mapping,
    brand_controls=BRAND,
    recipe_specification={"recipes": RECIPES},
    estimate=True,  # quote only
)
print("estimated_credits_consumed:", estimate.estimated_credits_consumed)  # illustrative
print("estimated_minutes:", estimate.estimated_minutes)

In [ ]:
run = client.datasets.run(
    dataset_id,
    column_mapping=column_mapping,
    brand_controls=BRAND,
    recipe_specification={"recipes": RECIPES},
    job_specification={
        "max_rows": 300,                  # pilot on a slice; raise for the full corpus
        "idempotency_key": "hf-import-pilot-1",
    },
    estimate=False,
)
print("run_id:", run.run_id)

In [ ]:
from adaption import DatasetTimeout

try:
    status = client.datasets.wait_for_completion(dataset_id, timeout=1800)
    print("run status:", status.status)
    if status.error:
        print("error:", status.error.message)
except DatasetTimeout as e:
    print("timed out after", e.timeout, "s; last status:", e.last_status)

## Step 5 - Read `improvement_percent`

Evaluation is polled separately from run status, exactly as in notebook 01.

In [ ]:
while True:
    ev = client.datasets.get_evaluation(dataset_id)
    if ev.status not in ("pending", "running"):
        break
    print("evaluation", ev.status, "... waiting")
    time.sleep(5)

if ev.status == "succeeded" and ev.quality:
    print("improvement_percent:", ev.quality.improvement_percent)  # the headline number
else:
    print("evaluation status:", ev.status)

## Recap

- HF import is **async**: we waited for `row_count is not None` before adapting.
- We kept the **completion only** anchor to stay safe from prompt-keyed deduplication.
- Estimate -> capped pilot -> read `improvement_percent` from `get_evaluation`.

Next: **03_publish.ipynb** to download the adapted dataset and release it to Hugging Face and Kaggle.